# Module 318 — GoodWords Challenge

This notebook attacks the authorized HTB Academy GoodWords challenge using only its documented `/challenge`, `/predict`, and `/submit` endpoints. The target chooses the base spam message and maximum added-word count. The notebook probes candidate words, appends the most promising ones, stops when the service reports ham, validates the exact suffix, and submits only a successful candidate.

**Target setup:** start the challenge instance in Academy, then set `TARGET_IP` and `TARGET_PORT` below (or set the `BASE_URL` environment variable). The endpoint is temporary; the notebook deliberately does not store a live target address or flag.

## The score change the code measures

For base message `x` and candidate word `w`, the single-word impact is `Δp(w) = p_spam(x) - p_spam(x + space + w)`. Read `Δp(w)` aloud as **“delta p of w.”** It is the original spam probability minus the probability after appending `w`. If the scores are `0.91` and `0.76`, then the measured change is `0.15`, or 15 percentage points. A positive impact lowers the score, but only the service's returned label confirms that the message now counts as ham.

The notebook keeps the original text fixed, joins added words with spaces, and uses the server's `max_added_words` value. The challenge states that validation uses letter-only tokens, so we explicitly count `[A-Za-z]+` tokens in the appended suffix before submitting.

In [ ]:
import os
import re

import requests
import pandas as pd

# Replace these placeholders after starting the HTB target, or set BASE_URL.
TARGET_IP = '<TARGET_IP>'
TARGET_PORT = '<PORT>'
BASE_URL = os.getenv(
    'BASE_URL', f'http://{TARGET_IP}:{TARGET_PORT}'
).rstrip('/')
REQUEST_TIMEOUT_SECONDS = 15
MAX_SINGLE_WORD_PROBES = 40

if '<TARGET_IP>' in BASE_URL or '<PORT>' in BASE_URL:
    raise ValueError(
        'Set TARGET_IP and TARGET_PORT, or define BASE_URL for the active instance.'
    )

session = requests.Session()
prediction_query_count = 0
query_log = []

## API helpers and server-provided constraints

`raise_for_status()` turns an HTTP error into a clear exception instead of treating a server error page as a model score. The prediction helper checks that the JSON includes both `label` and a probability in the expected 0-to-1 range. Each successful `/predict` request is counted once; `/challenge` and the final `/submit` are tracked separately. The current target must be online and reachable through the Academy connection before these cells can run.

In [ ]:
def get_json(path):
    response = session.get(
        f'{BASE_URL}{path}', timeout=REQUEST_TIMEOUT_SECONDS
    )
    response.raise_for_status()
    return response.json()


def predict(text):
    """Submit one text to the score endpoint and record its returned score."""
    global prediction_query_count
    response = session.post(
        f'{BASE_URL}/predict',
        json={'text': text},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    result = response.json()
    if 'label' not in result or 'spam_probability' not in result:
        raise ValueError(f'Unexpected /predict response fields: {list(result)}')

    probability = float(result['spam_probability'])
    if not 0.0 <= probability <= 1.0:
        raise ValueError(f'Invalid spam probability: {probability}')

    prediction_query_count += 1
    query_log.append({
        'query_number': prediction_query_count,
        'text_characters': len(text),
        'label': str(result['label']).lower(),
        'spam_probability': probability,
    })
    result['spam_probability'] = probability
    result['label'] = str(result['label']).lower()
    return result


challenge = get_json('/challenge')
base_message = challenge['base_message']
max_added_words = int(challenge['max_added_words'])
target_label = str(challenge.get('target_label', 'ham')).lower()
if target_label != 'ham':
    raise ValueError(f'This notebook expects target_label=ham, got {target_label!r}')
if max_added_words <= 0:
    raise ValueError('The challenge reported no available added-word budget')

print('Base message:', base_message)
print('Target label:', target_label)
print('Maximum added words:', max_added_words)

## Check the baseline before spending probes

The challenge endpoint selects a strong spam example. We still measure it ourselves to record the starting score and verify that `/predict` is working. A transport failure, malformed response, or unexpected label should be fixed before word-ranking results are interpreted.

In [ ]:
health_response = session.get(
    f'{BASE_URL}/health', timeout=REQUEST_TIMEOUT_SECONDS
)
health_response.raise_for_status()
print('Health response:', health_response.text[:300])

baseline_result = predict(base_message)
base_spam_probability = baseline_result['spam_probability']
print('Baseline prediction:', baseline_result)
if baseline_result['label'] != 'spam':
    print('Note: the service did not label the base message spam; confirm the active target and challenge state.')

## Probe candidate words

This compact list contains ordinary, legitimate-looking English terms like the Academy's scaffold. Every candidate must be one letter-only token, so the added-word count has an unambiguous meaning. We cap probes at 40 by default; lower that limit if the target imposes a stricter request policy. The ranking is based on this challenge's returned scores, not on the locally trained model or its white-box probability table.

In [ ]:
candidate_words = [
    'please', 'thanks', 'meeting', 'tomorrow', 'coffee', 'home', 'support',
    'good', 'great', 'safe', 'hello', 'friend', 'family', 'love', 'morning',
    'evening', 'call', 'later', 'today', 'work', 'team', 'update', 'schedule',
    'happy', 'available', 'appreciate', 'kindly', 'regards', 'cheers', 'understood',
    'message', 'help', 'welcome', 'wonderful', 'hope', 'talk', 'visit', 'lunch',
    'weekend', 'confirm',
]

# Remove duplicates but preserve the educational list's order.
candidate_words = list(dict.fromkeys(candidate_words))
invalid_words = [
    word for word in candidate_words
    if re.fullmatch(r'[A-Za-z]+', word) is None
]
if invalid_words:
    raise ValueError(f'Candidates must be letter-only single tokens: {invalid_words}')
print('Valid candidate words:', len(candidate_words))

In [ ]:
probe_words = candidate_words[:MAX_SINGLE_WORD_PROBES]
word_impacts = []
for word in probe_words:
    augmented = base_message + ' ' + word
    result = predict(augmented)
    impact = base_spam_probability - result['spam_probability']
    word_impacts.append({
        'word': word,
        'spam_probability': result['spam_probability'],
        'label': result['label'],
        'impact': impact,
    })

word_impact_df = pd.DataFrame(word_impacts).sort_values(
    'impact', ascending=False, ignore_index=True
)
display(word_impact_df)
print(f'Score queries used for individual probes: {prediction_query_count}')

A row's `impact` is the finite difference for one word against the fixed base. The output is sorted from largest positive score reduction to smallest. A score can go up, down, or stay the same; do not assume a word helps just because it sounds legitimate. The next step adds words one at a time and checks the returned label after every prefix, because the impact of a word can change in context.

In [ ]:
LETTER_TOKEN = re.compile(r'[A-Za-z]+')

def count_added_letter_tokens(original, augmented):
    """Verify append-only form and count letter-only tokens in the suffix."""
    if not augmented.startswith(original):
        raise ValueError('Submission must preserve the exact original base message')
    suffix = augmented[len(original):]
    if not suffix:
        return 0
    if re.fullmatch(r'(?: [A-Za-z]+)+', suffix) is None:
        raise ValueError('Added text must be space-separated letter-only tokens')
    return len(LETTER_TOKEN.findall(suffix))


positive_candidates = [
    row['word'] for row in word_impacts if row['impact'] > 0
]
# Keep the strongest positive singles first; the max token count is checked
# again while growing the message, before any final submission.
rank_limit = max(2 * max_added_words, 20)
ranked_candidates = positive_candidates[:rank_limit]
if not ranked_candidates:
    # If no isolated word helps, keep the best measured candidates as a small
    # follow-up experiment; interactions can differ from single-word effects.
    ranked_candidates = word_impact_df['word'].head(
        min(max_added_words, len(word_impact_df))
    ).tolist()

augmented_text = base_message
added_words = []
last_prediction = baseline_result
for word in ranked_candidates:
    if len(added_words) >= max_added_words:
        break
    proposed_text = augmented_text + ' ' + word
    proposed_count = count_added_letter_tokens(base_message, proposed_text)
    if proposed_count > max_added_words:
        raise RuntimeError('Refusing to exceed the server-provided word budget')

    last_prediction = predict(proposed_text)
    # Keep the candidate only after it was measured. The page's scaffold
    # appends ranked candidates sequentially and checks each growing prefix.
    augmented_text = proposed_text
    added_words.append(word)
    print(
        f'Added {len(added_words):2d}/{max_added_words}: {word:12} '
        f"spam={last_prediction['spam_probability']:.4f} "
        f"label={last_prediction['label']}"
    )
    if last_prediction['label'] == target_label:
        break

final_added_count = count_added_letter_tokens(base_message, augmented_text)
print('Final label:', last_prediction['label'])
print('Added tokens:', final_added_count, '/', max_added_words)
print('Augmented text:', augmented_text)

## Submit only after local checks pass

The last cell sends `/submit` only if the most recent `/predict` result is ham, the original message is unchanged, and the letter-token count is within the target's maximum. The service response is authoritative: inspect `result`, `details`, and `flag`. If the label is still spam, the cell will refuse to submit; adjust the candidate list or probe cap and rerun the construction cells against the still-active target.

In [ ]:
if last_prediction['label'] != target_label:
    raise RuntimeError(
        'Not submitting: the latest prediction has not reached the target label.'
    )

validated_count = count_added_letter_tokens(base_message, augmented_text)
if validated_count > max_added_words:
    raise RuntimeError('Not submitting: the appended-token budget is exceeded.')
if not augmented_text.startswith(base_message):
    raise RuntimeError('Not submitting: append-only validation failed.')

submission_response = session.post(
    f'{BASE_URL}/submit',
    json={'augmented_text': augmented_text},
    timeout=REQUEST_TIMEOUT_SECONDS,
)
submission_response.raise_for_status()
submission_result = submission_response.json()
print(submission_result)

## COAE recall

- A black-box prediction endpoint is an oracle: submit a message, observe label and score, and infer useful input changes from output differences.
- `Δp(w)=p_spam(base)-p_spam(base+w)` is the single-word score change. Positive means lower spam probability; only the target label means the classifier changed classes.
- The server's base message must remain an exact prefix. Add a space and letter-only tokens; count those tokens using the challenge's rule and stay within `max_added_words`.
- Single-word ranks are context-dependent. Measure the growing prefix and stop when the endpoint reports ham.
- Keep the active `BASE_URL` outside the committed notebook, use request timeouts, and submit only after the append-only and word-count checks pass.